In [ ]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


# Correlation matrix + directiveness composite ranking

Three cross-cutting views:

1. **Cross-metric correlation** — Pearson r across ~30 per-file metrics (mood, register quantitative, stance polarity, modality, vocab, loudness, sentence_register). Positive correlations in red, negative in blue, centered at zero.

2. **Top-25 most directive prompts** — extended composite z-score:
   `z(mood_marker_pct) + z(hard_prohibitions_pct) + z(caps_imp_pct) + z(directive_sent_pct) + z(configuring_sent_pct) − z(collaborative_sent_pct) − z(permissive_sent_pct) − z(appreciative_sent_pct)`.
   Higher = more directive. The three "soft" classes subtract because they soften authoritative language.

3. **Per-word vs per-sentence comparison** — six representative metrics shown side-by-side in their two unit views (`% of words` vs `per-sentence rate`), to clarify how the same signal looks under different normalizations.


### Cross-metric correlation + per-file directiveness ranking

Two final views:

- **Correlation matrix** across the main per-file metrics. Surfaces which dimensions move
  together (e.g. imperative density should correlate with deontic modality and prohibitions)
  and which are independent.
- **Top-25 most directive files** by a composite score: imperative-marker density + hard
  prohibitions density + CAPS imperative density (each z-scored, then summed). Hover any row
  for the full path.

In [ ]:
"""Cross-metric correlation matrix across the main per-file numeric metrics.

Updated for the new schema:
  - dropped: mood_imperative_sent_pct (now in sentence_register), evaluative_pct
  - added:   positive_evaluative_pct, negative_evaluative_pct (stance polarity split)
  - added:   the six sentence_register *_sent_pct classes
"""
import numpy as np

corr_cols = [
    # mood / lexical density
    "mood_marker_pct",
    # register quantitative
    "ttr", "mean_sent_len", "dep_depth", "f_score",
    # stance (5-class)
    "directive_pct", "expository_pct",
    "positive_evaluative_pct", "negative_evaluative_pct",
    "dialogic_pct",
    # modality
    "deontic_pct", "epistemic_pct", "dynamic_pct",
    # vocab
    "hard_prohibitions_pct", "hard_prescriptions_pct",
    "soft_prescriptions_pct", "hedging_pct", "structural_markers_pct",
    "pronouns_2p_pct", "pronouns_1p_pct",
    # loudness + reasoning
    "all_caps_pct", "caps_imp_pct",
    "just_pct", "just_ratio",
    # sentence_register (per-sentence, multi-label)
    "collaborative_sent_pct", "permissive_sent_pct", "appreciative_sent_pct",
    "imperative_sent_pct", "directive_sent_pct", "configuring_sent_pct",
]
corr = alt_df[corr_cols].corr().reset_index().melt(
    id_vars="index", var_name="other", value_name="r")
corr.columns = ["x", "y", "r"]

corr_chart = (
    alt.Chart(corr)
    .mark_rect()
    .encode(
        x=alt.X("x:N", sort=corr_cols, title=None,
                axis=alt.Axis(labelAngle=-45, labelLimit=160)),
        y=alt.Y("y:N", sort=corr_cols, title=None,
                axis=alt.Axis(labelLimit=160)),
        color=alt.Color("r:Q",
                         scale=alt.Scale(scheme="redblue", domain=[-1, 1], reverse=True),
                         title="Pearson r"),
        tooltip=[alt.Tooltip("x:N"), alt.Tooltip("y:N"),
                 alt.Tooltip("r:Q", format=".2f")],
    )
    .properties(width=620, height=620,
                title=f"Cross-metric correlation (Pearson r) — {len(corr_cols)} per-file metrics")
)
corr_chart


In [ ]:
"""Top-25 most directive files by extended composite z-score.

Updated formula adds the new sentence_register signals:
  + directive_sent_pct      (raises authority)
  + configuring_sent_pct    (raises authority)
  - collaborative_sent_pct  (softens — interpersonal)
  - permissive_sent_pct     (softens — reader-agency / "you can")
  - appreciative_sent_pct   (softens — gratitude / praise)
"""

def zscore(s):
    s = s.astype(float)
    return (s - s.mean()) / (s.std(ddof=0) or 1.0)

alt_df["directiveness"] = (
    zscore(alt_df["mood_marker_pct"])
    + zscore(alt_df["hard_prohibitions_pct"])
    + zscore(alt_df["caps_imp_pct"])
    + zscore(alt_df["directive_sent_pct"])
    + zscore(alt_df["configuring_sent_pct"])
    - zscore(alt_df["collaborative_sent_pct"])
    - zscore(alt_df["permissive_sent_pct"])
    - zscore(alt_df["appreciative_sent_pct"])
)

top25 = (alt_df.nlargest(25, "directiveness")
              [["path", "category", "n_tokens", "directiveness",
                "mood_marker_pct", "hard_prohibitions_pct",
                "caps_imp_pct",
                "directive_sent_pct", "configuring_sent_pct",
                "collaborative_sent_pct", "permissive_sent_pct",
                "appreciative_sent_pct",
                "just_ratio"]]
              .reset_index(drop=True))

print(f"composite directiveness range: "
      f"{alt_df['directiveness'].min():.2f}  ↔  {alt_df['directiveness'].max():.2f}")
display(top25.style.background_gradient(subset=["directiveness"], cmap="Reds")
                   .format({"directiveness": "{:.2f}",
                            "mood_marker_pct": "{:.2f}",
                            "hard_prohibitions_pct": "{:.2f}",
                            "caps_imp_pct": "{:.2f}",
                            "directive_sent_pct": "{:.1f}",
                            "configuring_sent_pct": "{:.1f}",
                            "collaborative_sent_pct": "{:.1f}",
                            "permissive_sent_pct": "{:.1f}",
                            "appreciative_sent_pct": "{:.1f}",
                            "just_ratio": "{:.2f}"}))

top25_chart = (
    alt.Chart(top25)
    .mark_bar()
    .encode(
        x=alt.X("directiveness:Q", title="Composite z-score (higher = more directive)"),
        y=alt.Y("path:N", sort="-x", title=None,
                axis=alt.Axis(labelLimit=320)),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=_cat_domain, range=_cat_range)),
        tooltip=[
            alt.Tooltip("path:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q", format=","),
            alt.Tooltip("directiveness:Q", format=".2f"),
            alt.Tooltip("mood_marker_pct:Q",
                         title="imperative markers %", format=".2f"),
            alt.Tooltip("hard_prohibitions_pct:Q",
                         title="hard prohibitions %", format=".2f"),
            alt.Tooltip("caps_imp_pct:Q",
                         title="CAPS imperative %", format=".2f"),
            alt.Tooltip("directive_sent_pct:Q",
                         title="directive sent %", format=".1f"),
            alt.Tooltip("configuring_sent_pct:Q",
                         title="configuring sent %", format=".1f"),
            alt.Tooltip("collaborative_sent_pct:Q",
                         title="collab sent %", format=".1f"),
            alt.Tooltip("permissive_sent_pct:Q",
                         title="permissive sent %", format=".1f"),
            alt.Tooltip("just_ratio:Q", title="justification ratio", format=".2f"),
        ],
    )
    .properties(width=560, height=560,
                title="Top 25 most directive prompts (extended z-summed composite)")
)
top25_chart


### Per-word vs per-sentence comparison

Every density metric is now reported in two units. The chart below puts them side-by-side for
the most informative metrics, so you can see how the two views differ. Categories with longer
sentences (e.g. *Tool description* at ~40 tokens/sent) show a much higher per-sentence rate
than per-word percentage, while categories with short sentences (e.g. *System reminder* at
~32 tokens/sent) sit closer together.

In [ ]:
"""Per-word vs per-sentence comparison: side-by-side bars for key metrics."""

# Six representative metrics (one per dimension), with their (pct_col, per_sent_col) pair
metric_pairs = [
    ("Imperative markers",  "mood_marker_pct",        "mood_marker_per_sent"),
    ("Hard prohibitions",   "hard_prohibitions_pct",  "hard_prohibitions_per_sent"),
    ("Directive stance",    "directive_pct",          "directive_per_sent"),
    ("Hedging",             "hedging_pct",            "hedging_per_sent"),
    ("CAPS imperative",     "caps_imp_pct",           "caps_imp_per_sent"),
    ("Justifications",      "just_pct",               "just_per_sent"),
]

rows = []
for cat in cats:
    sub = alt_df[alt_df["category"] == cat]
    for label, pct_col, ps_col in metric_pairs:
        rows.append({"category": cat, "metric": label, "unit": "% of words",
                     "value": round(sub[pct_col].mean(), 4)})
        rows.append({"category": cat, "metric": label, "unit": "per sentence",
                     "value": round(sub[ps_col].mean(), 4)})
ws_df = pd.DataFrame(rows)

unit_sel = alt.selection_point(fields=["unit"], bind="legend")

ws_chart = (
    alt.Chart(ws_df)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None),
        y=alt.Y("category:N", sort="-x", title=None),
        color=alt.Color("unit:N",
                         scale=alt.Scale(range=["#4e79a7", "#e15759"]),
                         legend=alt.Legend(title="Unit", orient="bottom")),
        opacity=alt.condition(unit_sel, alt.value(0.85), alt.value(0.15)),
        row=alt.Row("metric:N", title=None,
                     sort=[label for label, _, _ in metric_pairs],
                     header=alt.Header(labelAngle=0, labelAlign="left")),
        column=alt.Column("unit:N", title=None,
                           sort=["% of words", "per sentence"]),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("metric:N"),
            alt.Tooltip("unit:N"),
            alt.Tooltip("value:Q", format=".3f"),
        ],
    )
    .add_params(unit_sel)
    .properties(width=180, height=120,
                title="Six metrics, two unit views (per-word % vs per-sentence rate)")
    .resolve_scale(x="independent")
)
ws_chart
